#Harry Potter RAG Notebook

#Installing packages

This section installs all the necessary Python packages for this RAG pipeline, including `pymupdf` for PDF processing, `sentence-transformers` for embedding generation, and `qdrant-client` for vector database operations.

In [1]:
!pip install pymupdf
!pip install -q sentence-transformers
!pip install qdrant-client


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 12.3 MB/s eta 0:00:00


# Importing Libraries

Here, we import all the required libraries and modules that will be used throughout the notebook for tasks such as file system operations, text cleaning, embedding, and interacting with the Qdrant vector database.

In [2]:
from google.colab import drive
import pymupdf
import os
import re
from collections import Counter
from sentence_transformers import SentenceTransformer
import numpy as np
from pathlib import Path
import os
from dotenv import load_dotenv
import json
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct

To access files stored in Google Drive, we first mount Google Drive to the Colab environment. This allows the notebook to read and write files from your connected Drive account.

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In this section, we define important directory paths for our project, including the base project directory in Google Drive, the path to the input PDF file, and the directory where processed data will be stored.

In [4]:
PROJECT_DIR = "/content/drive/MyDrive/RAG"

PDF_PATH = f"{PROJECT_DIR}/Data/harrypotter.pdf"

DATASET_DIR = f"{PROJECT_DIR}/dataset"

print("Project directory:", PROJECT_DIR)
print("PDF path:", PDF_PATH)
print("Dataset directory:", DATASET_DIR)

Project directory: /content/drive/MyDrive/RAG
PDF path: /content/drive/MyDrive/RAG/Data/harrypotter.pdf
Dataset directory: /content/drive/MyDrive/RAG/dataset


We open the PDF document using `pymupdf` and display the total number of pages to confirm that the document has been loaded correctly.

In [5]:
doc = pymupdf.open(PDF_PATH)
print("Number of pages:", len(doc))

Number of pages: 3623


This code block extracts text from each page of the PDF document and saves it into a single Markdown file (`output.md`). Each page's content is prefaced with a '## Page X' header for easy parsing later.

In [6]:
output_md = f"{PROJECT_DIR}/output.md"

with open(output_md, "w", encoding="utf-8") as f:
    for page_num, page in enumerate(doc, start=1):
        text = page.get_text()

        f.write(f"## Page {page_num}\n\n")
        f.write(text)
        f.write("\n\n")

print("Saved:", output_md)

Saved: /content/drive/MyDrive/RAG/output.md


We read and print a preview of the `output.md` file to visually inspect the extracted text and ensure that the content has been correctly saved and formatted. This helps in early detection of any parsing issues.

In [7]:
with open(output_md, "r", encoding="utf-8") as f:
    preview = f.read(3000)

print(preview)

## Page 1



## Page 2



## Page 3



## Page 4



## Page 5



## Page 6

CONTENTS
Harry Potter and the Sorcerer’s Stone
Harry Potter and the Chamber of Secrets
Harry Potter and the Prisoner of Azkaban
Harry Potter and the Goblet of Fire
Harry Potter and the Order of the Phoenix
Harry Potter and the Half-Blood Prince
Harry Potter and the Deathly Hallows


## Page 7



## Page 8



## Page 9

 
FOR JESSICA, WHO LOVES STORIES,
FOR ANNE, WHO LOVED THEM TOO;
AND FOR DI, WHO HEARD THIS ONE FIRST.


## Page 10

 
CONTENTS
ONE
The Boy Who Lived
TWO
The Vanishing Glass
THREE
The Letters from No One
FOUR
The Keeper of the Keys
FIVE
Diagon Alley
SIX
The Journey from Platform Nine and Three-quarters
SEVEN
The Sorting Hat
EIGHT
The Potions Master
NINE
The Midnight Duel
TEN
Halloween
ELEVEN
Quidditch
TWELVE


## Page 11

The Mirror of Erised
THIRTEEN
Nicolas Flamel
FOURTEEN
Norbert the Norwegian Ridgeback
FIFTEEN
The Forbidden Forest
SIXTEEN
Through the Trapdoor
SEVENTEEN
The Man with Two Faces



Here, we define a regular expression to identify page boundaries within the `output.md` file and then parse the file to extract individual pages. Each page's text and number are stored in a list of dictionaries.

In [8]:
page_pattern = r"## Page (\d+)\n"

# Read the entire content of output_md into the 'text' variable
with open(output_md, "r", encoding="utf-8") as f:
    text = f.read()

matches = list(re.finditer(page_pattern, text))

pages = []

for i, match in enumerate(matches):
    page_number = int(match.group(1))

    start = match.end()

    if i + 1 < len(matches):
        end = matches[i + 1].start()
    else:
        end = len(text)

    page_text = text[start:end].strip()

    pages.append({
        "page": page_number,
        "text": page_text
    })

print("Number of pages extracted:", len(pages))
print("\nFirst page:")
print(pages[0])

print("\nPage 12:")
print(pages[11])

Number of pages extracted: 3623

First page:
{'page': 1, 'text': ''}

Page 12:
{'page': 12, 'text': 'M\n \nCHAPTER  ONE\nTHE BOY WHO LIVED\nr. and Mrs. Dursley, of number four, Privet Drive, were proud to say\nthat they were perfectly normal, thank you very much. They were the\nlast people you’d expect to be involved in anything strange or mysterious,\nbecause they just didn’t hold with such nonsense.\nMr. Dursley was the director of a firm called Grunnings, which made drills.\nHe was a big, beefy man with hardly any neck, although he did have a very\nlarge mustache. Mrs. Dursley was thin and blonde and had nearly twice the\nusual amount of neck, which came in very useful as she spent so much of her\ntime craning over garden fences, spying on the neighbors. The Dursleys had a\nsmall son called Dudley and in their opinion there was no finer boy anywhere.\nThe Dursleys had everything they wanted, but they also had a secret, and\ntheir greatest fear was that somebody would discover it. Th

This section defines the page ranges for each book in the Harry Potter series. This mapping is essential for associating each extracted page with its corresponding book title, enabling book-specific queries later.

In [9]:
BOOK_RANGES = [
    (12, 274, "Harry Potter and the Sorcerer's Stone"),
    (282, 565, "Harry Potter and the Chamber of Secrets"),
    (573, 939, "Harry Potter and the Prisoner of Azkaban"),
    (949, 1560, "Harry Potter and the Goblet of Fire"),
    (1570, 2406, "Harry Potter and the Order of the Phoenix"),
    (2409, 2964, "Harry Potter and the Half-Blood Prince"),
    (2974, 3622, "Harry Potter and the Deathly Hallows"),
]

The `get_book` function is a helper that determines which Harry Potter book a given page number belongs to, based on the `BOOK_RANGES` defined previously. This function will be applied to each page.

In [10]:
def get_book(page_number):
    for start, end, book in BOOK_RANGES:
        if start <= page_number <= end:
            return book

    return None

We iterate through the list of pages and assign the correct book title to each page using the `get_book` function. This enriches our page data with book metadata, which is crucial for contextual retrieval.

In [11]:
for page in pages:
    page["book"] = get_book(page["page"])

print(pages[11])
print(pages[274 - 1])
print(pages[281])

{'page': 12, 'text': 'M\n \nCHAPTER  ONE\nTHE BOY WHO LIVED\nr. and Mrs. Dursley, of number four, Privet Drive, were proud to say\nthat they were perfectly normal, thank you very much. They were the\nlast people you’d expect to be involved in anything strange or mysterious,\nbecause they just didn’t hold with such nonsense.\nMr. Dursley was the director of a firm called Grunnings, which made drills.\nHe was a big, beefy man with hardly any neck, although he did have a very\nlarge mustache. Mrs. Dursley was thin and blonde and had nearly twice the\nusual amount of neck, which came in very useful as she spent so much of her\ntime craning over garden fences, spying on the neighbors. The Dursleys had a\nsmall son called Dudley and in their opinion there was no finer boy anywhere.\nThe Dursleys had everything they wanted, but they also had a secret, and\ntheir greatest fear was that somebody would discover it. They didn’t think\nthey could bear it if anyone found out about the Potters. Mrs.

The `clean_text` function is designed to preprocess the raw text extracted from the PDF pages. It performs normalization of line endings, removes extra spaces, and reduces excessive blank lines to ensure consistent and clean text for embedding.

In [12]:
def clean_text(text):
    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove trailing spaces from each line
    text = "\n".join(line.rstrip() for line in text.split("\n"))

    # Reduce excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Reduce repeated spaces
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

This cell applies the `clean_text` function to every page's text content, creating a new `clean_text` field for each page. This standardized text will be used for generating embeddings.

In [13]:
for page in pages:
    page["clean_text"] = clean_text(page["text"])

We display a comparison between the original and cleaned text for a sample page to visually confirm that the `clean_text` function is working as intended.

In [14]:
print("ORIGINAL:\n")
print(pages[11]["text"][:1000])

print("\n\nCLEANED:\n")
print(pages[11]["clean_text"][:1000])

ORIGINAL:

M
 
CHAPTER  ONE
THE BOY WHO LIVED
r. and Mrs. Dursley, of number four, Privet Drive, were proud to say
that they were perfectly normal, thank you very much. They were the
last people you’d expect to be involved in anything strange or mysterious,
because they just didn’t hold with such nonsense.
Mr. Dursley was the director of a firm called Grunnings, which made drills.
He was a big, beefy man with hardly any neck, although he did have a very
large mustache. Mrs. Dursley was thin and blonde and had nearly twice the
usual amount of neck, which came in very useful as she spent so much of her
time craning over garden fences, spying on the neighbors. The Dursleys had a
small son called Dudley and in their opinion there was no finer boy anywhere.
The Dursleys had everything they wanted, but they also had a secret, and
their greatest fear was that somebody would discover it. They didn’t think
they could bear it if anyone found out about the Potters. Mrs. Potter was Mrs.
Dursley’s 

This section filters the `pages` list to create a new list, `non_empty_pages`, which only includes pages that have actual content after cleaning. This helps in removing metadata pages or blank pages from consideration.

In [15]:
non_empty_pages = [
    page for page in pages
    if page["clean_text"]
]

print("Original pages:", len(pages))
print("Non-empty pages:", len(non_empty_pages))
print("Empty pages:", len(pages) - len(non_empty_pages))

Original pages: 3623
Non-empty pages: 3604
Empty pages: 19


Here, we further refine our page list by creating `book_pages`, which includes only those pages that have both a valid book assignment and non-empty cleaned text. This ensures we work with relevant and well-attributed content.

In [16]:
book_pages = [
    page for page in pages
    if page["book"] is not None and page["clean_text"]
]

print("Book pages:", len(book_pages))

Book pages: 3567


This cell provides a summary of the number of pages at various stages of processing: original, non-empty, and those assigned to specific books. This gives an overview of the data filtering process.

In [17]:
print("Original pages:", len(pages))
print("Non-empty pages:", len(non_empty_pages))
print("Book pages:", len(book_pages))
print("Empty pages:", len(pages) - len(non_empty_pages))

Original pages: 3623
Non-empty pages: 3604
Book pages: 3567
Empty pages: 19


Using the `collections.Counter`, we calculate and display the number of pages attributed to each Harry Potter book. This gives a breakdown of the content distribution across the series.

In [18]:
book_counts = Counter(page["book"] for page in book_pages)

for book, count in book_counts.items():
    print(f"{book}: {count} pages")

Harry Potter and the Sorcerer's Stone: 263 pages
Harry Potter and the Chamber of Secrets: 284 pages
Harry Potter and the Prisoner of Azkaban: 367 pages
Harry Potter and the Goblet of Fire: 612 pages
Harry Potter and the Order of the Phoenix: 837 pages
Harry Potter and the Half-Blood Prince: 555 pages
Harry Potter and the Deathly Hallows: 649 pages


# Chunking

This section prepares the final `documents` list that will be used for embedding and storage in the vector database. Each document contains the page number, book title, and the cleaned text content.

In [19]:
documents = []

for page in book_pages:
    documents.append({
        "page": page["page"],
        "book": page["book"],
        "text": page["clean_text"]
    })

print("Documents to embed:", len(documents))

Documents to embed: 3567


A specific page (page 12) is retrieved from the `documents` list to verify its content and associated metadata. This acts as a spot check to ensure the document preparation is accurate.

In [20]:
page_12 = next(doc for doc in documents if doc["page"] == 12)

print(page_12)

{'page': 12, 'book': "Harry Potter and the Sorcerer's Stone", 'text': 'M\n\nCHAPTER ONE\nTHE BOY WHO LIVED\nr. and Mrs. Dursley, of number four, Privet Drive, were proud to say\nthat they were perfectly normal, thank you very much. They were the\nlast people you’d expect to be involved in anything strange or mysterious,\nbecause they just didn’t hold with such nonsense.\nMr. Dursley was the director of a firm called Grunnings, which made drills.\nHe was a big, beefy man with hardly any neck, although he did have a very\nlarge mustache. Mrs. Dursley was thin and blonde and had nearly twice the\nusual amount of neck, which came in very useful as she spent so much of her\ntime craning over garden fences, spying on the neighbors. The Dursleys had a\nsmall son called Dudley and in their opinion there was no finer boy anywhere.\nThe Dursleys had everything they wanted, but they also had a secret, and\ntheir greatest fear was that somebody would discover it. They didn’t think\nthey could bear

# Embedding

Here, we load the `SentenceTransformer` model, specifically `intfloat/multilingual-e5-large`, which will be used to convert text content into numerical embeddings. This model is chosen for its multilingual capabilities and strong performance.

In [21]:
MODEL_NAME = "intfloat/multilingual-e5-large"

embedding_model = SentenceTransformer(MODEL_NAME)

print("Embedding model loaded!")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Embedding model loaded!


This cell performs a quick test of the loaded embedding model by encoding a sample text from the first document. It prints the type and dimension of the resulting embedding to confirm the model's functionality.

In [22]:
test_text = documents[0]["text"]

test_embedding = embedding_model.encode(
    f"passage: {test_text}"
)
print("Embedding dimension:", len(test_embedding))

Embedding dimension: 1024


We prepare a list of all cleaned text snippets, formatted as 'passage: {text}', which is the required input format for the `multilingual-e5-large` model. This list will be used for bulk embedding.

In [23]:
texts = [
    f"passage: {doc['text']}"
    for doc in documents
]

print("Number of texts:", len(texts))

Number of texts: 3567


This code block generates embeddings for all the prepared text snippets in batches. Batch processing improves efficiency, and `normalize_embeddings=True` ensures that all vectors are unit length, which is beneficial for cosine similarity calculations.

In [24]:
embeddings = embedding_model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True  #Normalization makes the vectors unit length, which makes cosine-based comparison straightforward
)

print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/223 [00:00<?, ?it/s]

Embeddings shape: (3567, 1024)


The generated embeddings are saved to a `.npy` file on Google Drive. This allows for persistent storage of the embeddings, so they don't need to be recomputed if the notebook session restarts.

In [25]:
EMBEDDINGS_PATH = Path(PROJECT_DIR) / "embeddings.npy"

np.save(EMBEDDINGS_PATH, embeddings)

print("Embeddings saved to:")
print(EMBEDDINGS_PATH)
print("Shape:", embeddings.shape)

Embeddings saved to:
/content/drive/MyDrive/RAG/embeddings.npy
Shape: (3567, 1024)


The `documents` list, containing all the parsed and cleaned page data, is saved as a JSON file. This file will serve as a persistent record of the document metadata and content.

In [26]:
DOCUMENTS_PATH = Path(PROJECT_DIR) / "documents.json"

with open(DOCUMENTS_PATH, "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

print("Documents saved to:")
print(DOCUMENTS_PATH)
print("Number of documents:", len(documents))

Documents saved to:
/content/drive/MyDrive/RAG/documents.json
Number of documents: 3567


To ensure the documents were saved correctly, this cell loads the `documents.json` file back into a variable and prints the total number of loaded documents and the first document's content for verification.

In [27]:
with open(DOCUMENTS_PATH, "r", encoding="utf-8") as f:
    loaded_documents = json.load(f)

print("Loaded documents:", len(loaded_documents))
print(loaded_documents[0])

Loaded documents: 3567
{'page': 12, 'book': "Harry Potter and the Sorcerer's Stone", 'text': 'M\n\nCHAPTER ONE\nTHE BOY WHO LIVED\nr. and Mrs. Dursley, of number four, Privet Drive, were proud to say\nthat they were perfectly normal, thank you very much. They were the\nlast people you’d expect to be involved in anything strange or mysterious,\nbecause they just didn’t hold with such nonsense.\nMr. Dursley was the director of a firm called Grunnings, which made drills.\nHe was a big, beefy man with hardly any neck, although he did have a very\nlarge mustache. Mrs. Dursley was thin and blonde and had nearly twice the\nusual amount of neck, which came in very useful as she spent so much of her\ntime craning over garden fences, spying on the neighbors. The Dursleys had a\nsmall son called Dudley and in their opinion there was no finer boy anywhere.\nThe Dursleys had everything they wanted, but they also had a secret, and\ntheir greatest fear was that somebody would discover it. They didn’t

#Qdrant

This section focuses on configuring and interacting with Qdrant, a vector database. First, we load environment variables from the `.env` file, which typically contains API keys, URL, and collection names for secure access.

In [30]:
load_dotenv(f"{PROJECT_DIR}/.env" ,override=True)

QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
QDRANT_COLLECTION = os.getenv("QDRANT_COLLECTION")

print("URL loaded:", bool(QDRANT_URL))
print("API key loaded:", bool(QDRANT_API_KEY))
print("Collection:", QDRANT_COLLECTION)

URL loaded: True
API key loaded: True
Collection: harry_potter


In [31]:
client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)

print(client.get_collections())

collections=[CollectionDescription(name='harry_potter')]


This crucial step involves uploading all the generated embeddings and their corresponding document metadata to the Qdrant vector database. The data is processed in batches for efficiency, creating `PointStruct` objects for each document.

In [32]:
BATCH_SIZE = 100

for start in range(0, len(documents), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(documents))

    points = []

    for i in range(start, end):
        doc = documents[i]

        points.append(
            PointStruct(
                id=i,
                vector=embeddings[i].tolist(),
                payload={
                    "book_name": doc["book"],
                    "page_number": doc["page"],
                    "content": doc["text"],
                },
            )
        )

    client.upsert(
        collection_name=QDRANT_COLLECTION,
        points=points,
    )

    print(f"Uploaded {end}/{len(documents)}")

Uploaded 100/3567
Uploaded 200/3567
Uploaded 300/3567
Uploaded 400/3567
Uploaded 500/3567
Uploaded 600/3567
Uploaded 700/3567
Uploaded 800/3567
Uploaded 900/3567
Uploaded 1000/3567
Uploaded 1100/3567
Uploaded 1200/3567
Uploaded 1300/3567
Uploaded 1400/3567
Uploaded 1500/3567
Uploaded 1600/3567
Uploaded 1700/3567
Uploaded 1800/3567
Uploaded 1900/3567
Uploaded 2000/3567
Uploaded 2100/3567
Uploaded 2200/3567
Uploaded 2300/3567
Uploaded 2400/3567
Uploaded 2500/3567
Uploaded 2600/3567
Uploaded 2700/3567
Uploaded 2800/3567
Uploaded 2900/3567
Uploaded 3000/3567
Uploaded 3100/3567
Uploaded 3200/3567
Uploaded 3300/3567
Uploaded 3400/3567
Uploaded 3500/3567
Uploaded 3567/3567


After uploading, we perform a count of the points (documents) in the specified Qdrant collection to verify that all documents have been successfully indexed in the vector database.

In [33]:
count = client.count(
    collection_name=QDRANT_COLLECTION,
    exact=True
)

print("Points in Qdrant:", count.count)

Points in Qdrant: 3567


This section demonstrates how to perform a similarity search using the Qdrant client. A query is embedded, and then the vector database is queried to find the top 5 most relevant documents based on cosine similarity.

# Retrieval

In [36]:
query = "Who are Harry Potter's parents?"

query_vector = embedding_model.encode(
    [f"query: {query}"],
    normalize_embeddings=True,
)[0].tolist()

results = client.query_points(
    collection_name=QDRANT_COLLECTION,
    query=query_vector,
    limit=3,
    with_payload=True,
).points

for i, result in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Score:", result.score)
    print("Book:", result.payload["book_name"])
    print("Page:", result.payload["page_number"])
    print("Content:", result.payload["content"][:500])


--- Result 1 ---
Score: 0.82022446
Book: Harry Potter and the Deathly Hallows
Page: 3157
Content: The silent kitchen seemed to hum with the shock of the recent scene and
with Ron and Hermione’s unspoken reproaches. The Daily Prophet Lupin had
brought was still lying on the table, Harry’s own face staring up at the ceiling
from the front page. He walked over to it and sat down, opened the paper at
random, and pretended to read. He could not take in the words; his mind was
still too full of the encounter with Lupin. He was sure that Ron and Hermione
had resumed their silent communications on t

--- Result 2 ---
Score: 0.81909263
Book: Harry Potter and the Goblet of Fire
Page: 1452
Content: ask Neville this, in almost four years of knowing him.
“Yes, they were talking about Neville’s parents,” said Dumbledore. “His
father, Frank, was an Auror just like Professor Moody. He and his wife were
tortured for information about Voldemort’s whereabouts after he lost his
powers, as you heard.”
“So

We test the retrieval system using the question "Who are Harry Potter's parents?".

The retrieved results are relevant to Harry and his family, but the top 3 retrieved passages do not provide enough direct information to confidently answer the question. Therefore, the retrieval result alone is not sufficient for an accurate final answer.

In the complete RAG pipeline, the retrieved context is passed to the LLM, which can combine the relevant information from the retrieved passages to generate a more useful and accurate answer. If the required information is still not present in the context, the LLM should avoid inventing an answer.